# 🧹 Pandas III + NumPy — Limpeza e Transformação de Dados

---

## 🎯 Objetivos da Aula

Ao final desta aula, o aluno será capaz de:

1. **Identificar e diagnosticar problemas** em dados reais (valores faltantes, duplicatas, tipos incorretos) usando funções como `isna()`, `isnull()`, `duplicated()` e `info()`
2. **Remover dados problemáticos** com segurança, escolhendo entre `drop()`, `dropna()` e `drop_duplicates()` conforme o contexto
3. **Tratar valores faltantes** (NaN/None) usando diferentes estratégias: valores fixos, média, mediana, forward/backward fill
4. **Transformar e enriquecer colunas** usando `np.where()` e funções customizadas
5. **Aplicar filtros, substituições e acessos** precisos aos dados usando `loc[]`, `iloc[]`, `at[]` e `replace()`

---

## 🔥 Aquecimento — Problema Gerador
Imagine que você trabalha em uma startup de e-commerce. O time de vendas acabou de entregar para você um arquivo Excel com os dados de todos os clientes cadastrados no último trimestre: **5.000 linhas com nomes, e-mails, telefones, data de compra e valor gasto**.

Você abre o arquivo na primeira reunião com o CEO e descobre:
- 🚨 **Coluna "telefone" tem 1.200 valores em branco** — alguns clientes não forneceram
- 🚨 **Coluna "valor_gasto" tem "R$ 150" em texto**, não número — impossível fazer contas
- 🚨 **500 linhas estão duplicadas** — o mesmo cliente foi cadastrado 2 vezes por erro do sistema
- 🚨 **A coluna "data_compra" tem 300 valores faltantes** — o sistema antigo não registrou

O CEO olha para você e pergunta: *"Quanto nossos clientes gastaram realmente? Quantos clientes únicos temos?"*

Você não consegue responder porque os dados **não estão limpos**. Nesta aula, você aprenderá a:
- ✅ **Enxergar o problema** antes que ele vire lixo em uma análise
- ✅ **Limpar e preparar** os dados para análise real
- ✅ **Transformar valores** para o formato que você precisa
- ✅ **Tomar decisões conscientes** sobre o que manter, descartar ou corrigir

**Essa é a realidade de 80% do trabalho de um Data Scientist: não é modelagem com IA, é limpeza de dados.**

---

## 📖 Bloco Teórico + Analogias

### 🔍 Entendendo o Problema: Por Que Dados Precisam de Limpeza?

Pense em um **balcão de atendimento de um banco**. Os clientes chegam e preenchem fichas de cadastro:
- Alguns escrevem de caneta azul, outros de vermelha ❌
- Alguns colocam o CPF, outros deixam em branco ❌
- Alguns escrevem "São Paulo" e outros "SP" ❌
- Alguns repetem a mesma informação duas vezes ❌

O gerente do banco **não pode somar dinheiro** enquanto essas fichas estiverem bagunçadas. Ele precisa:
1. **Verificar** quais fichas estão incompletas ou erradas
2. **Descartar** duplicatas
3. **Padronizar** (ex: sempre "SP", nunca "São Paulo")
4. **Preencher buracos** (ex: tentar ligar para quem deixou telefone vazio)

**Dados sujos = fichas bagunçadas. Python + Pandas = ferramentas para organizar essas fichas.**

### 1️⃣ Identificação de Problemas: Diagnóstico Rápido


#### 📊 **Visão Geral com `shape`, `info()` e `describe()`**

A primeira coisa que você faz ao receber um dataset novo é **"radiografar"** ele — entender quantas linhas, quantas colunas, quais tipos de dados, e onde estão os problemas.

**Analogia:** como um médico que faz um raio-X antes de operar. Você precisa ver a estrutura antes de mexer.

In [1]:
import pandas as pd
import numpy as np

clientes = pd.read_csv('c:/Temp/clientes.csv')
display(clientes)

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.4,RS
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP
...,...,...,...,...,...,...,...
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ
5521,1579,Cliente_1578,cliente1578@mail.com,5.511998e+12,2024-02-28,386.22,BA


In [2]:
# Shape
print(clientes.shape)

(5523, 7)


In [3]:
# 2. info() — tipo de cada coluna e quantos valores não vazios tem
clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 5523 entries, 0 to 5522
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente_id   5523 non-null   int64  
 1   nome         5523 non-null   str    
 2   email        5349 non-null   str    
 3   telefone     4177 non-null   float64
 4   data_compra  5178 non-null   str    
 5   valor_gasto  5523 non-null   str    
 6   estado       5523 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 567.7 KB


**Leitura:** Note que `telefone` tem apenas 4.177 valores não-nulos de 5.523 = **1.346 vazios!**

In [4]:
# 3. describe() — estatísticas básicas (funciona bem com números)
clientes.describe()

,cliente_id,telefone
count,5523.000000,4.177000e+03
mean,2486.617418,5.511950e+12
std,1445.266426,2.886485e+07
min,1.000000,5.511900e+12
25%,1233.500000,5.511925e+12
50%,2484.000000,5.511949e+12
75%,3735.500000,5.511976e+12
max,5000.000000,5.512000e+12


---

#### 🔎 **Valores Faltantes: `isna()`, `isnull()`, `notnull()`**

Python chama valores vazios de **NaN** (Not a Number) ou **None**. Você precisa localizá-los antes de trabalhar.


In [11]:
# isna() — retorna True onde há valor faltante
clientes['telefone'].isna()


0       False
1       False
2       False
3       False
4       False
        ...  
5518    False
5519    False
5520    False
5521    False
5522     True
Name: telefone, Length: 5523, dtype: bool

In [9]:
# Quantos valores faltam em cada coluna?
print(clientes.isna().sum())

cliente_id        0
nome              0
email           174
telefone       1346
data_compra     345
valor_gasto       0
estado            0
dtype: int64


**Tradução:** 1.346 telefones faltam, 174 e-mails faltam, etc.

In [15]:
# isnull() — exatamente igual a isna()
# (são sinônimos — ambos funcionam)
print(clientes.isnull().sum())
print()
# notnull() — inverso de isna()
print(clientes.notnull().sum())

cliente_id        0
nome              0
email           174
telefone       1346
data_compra     345
valor_gasto       0
estado            0
dtype: int64

cliente_id     5523
nome           5523
email          5349
telefone       4177
data_compra    5178
valor_gasto    5523
estado         5523
dtype: int64


In [22]:
# Qual é a percentagem de dados faltantes por coluna?
percentual_faltantes = clientes.isna().sum()/len(clientes) * 100
percentual_faltantes.round(2)

cliente_id      0.00
nome            0.00
email           3.15
telefone       24.37
data_compra     6.25
valor_gasto     0.00
estado          0.00
dtype: float64


**Insight:** 24% dos telefones estão faltando — é um problema!

---
#### 📋 **Duplicatas: `duplicated()` e `drop_duplicates()`**

Duplicatas são **linhas repetidas** — o mesmo cliente cadastrado 2 vezes.

**Analogia:** como duas convites chegando para o mesmo convidado. Você quer contar pessoas, não convites.

In [24]:
# Quantas linhas são duplicadas (comparando TODAS as colunas)?
print(clientes.duplicated().sum())

523


In [28]:
# Quais são essas linhas duplicadas?
display(clientes[clientes.duplicated()])

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
206,1056,Cliente_1055,cliente1055@mail.com,NaN,2024-01-08,R$ 164.69,RJ
210,298,Cliente_297,cliente297@mail.com,NaN,NaN,R$ 152.44,SP
278,2032,Cliente_2031,cliente2031@mail.com,5.511977e+12,2024-03-14,343.4,RJ
325,30,Cliente_29,NaN,NaN,NaN,R$ 488.22,RS
404,3790,Cliente_3789,cliente3789@mail.com,5.511938e+12,2024-02-22,244.67,RJ
...,...,...,...,...,...,...,...
5505,3626,Cliente_3625,cliente3625@mail.com,5.511918e+12,2024-01-27,99.25,SP
5515,486,Cliente_485,cliente485@mail.com,NaN,2024-02-28,R$ 32.49,RJ
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ


In [31]:
# Às vezes você quer saber duplicatas só de UMA coluna
# Ex: quantos clientes únicos temos? (não contar o mesmo e-mail 2 vezes)

print(clientes.duplicated(subset=['email']).sum())

672


In [33]:
# Marca duplicatas de e-mail no DataFrame

# keep=False marca todas as linhas que aparecem mais de uma vez

display(clientes[clientes.duplicated(subset=['email'], keep=False)])



,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP
6,4583,Cliente_4582,cliente4582@mail.com,5.511902e+12,2024-02-25,280.67,RS
19,2844,Cliente_2843,cliente2843@mail.com,5.511919e+12,2024-03-08,122.74,MG
20,3050,Cliente_3049,cliente3049@mail.com,5.511994e+12,2024-03-29,334.43,MG
...,...,...,...,...,...,...,...
5510,131,Cliente_130,NaN,NaN,NaN,R$ 118.93,BA
5515,486,Cliente_485,cliente485@mail.com,NaN,2024-02-28,R$ 32.49,RJ
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ


In [ ]:
# keep='first' marca como duplicada a partir da segunda ocorrência
# keep='last' marca como duplicada a partir da primeira ocorrência




#### ✂️ **Remover Colunas com `drop()`**


In [ ]:

# Remove UMA coluna

# Remove VÁRIAS colunas


# Remove uma LINHA específica (por índice)

# Remove por LISTA de índices


In [ ]:
clientes_final.info()

In [ ]:
# Remove duplicatas considerando todas as colunas


In [ ]:
clientes_final

#### 🎯 **Análise de Variáveis: `unique()`, `nunique()`, `value_counts()`**

Quando você tem uma **coluna categórica** (ex: estado, categoria de produto), você quer saber:
- Quais são os valores únicos?
- Quantas categorias diferentes existem?
- Qual categoria aparece mais vezes?

In [ ]:

# unique() — lista todos os valores DIFERENTES


# nunique() — conta quantos valores diferentes existem


# value_counts() — conta quantas vezes cada valor aparece (do maior pro menor)


In [ ]:
# normalize=True — mostra em percentual ao invés de contagem


---

### 2️⃣ Remoção de Dados: `drop()`, `dropna()`, `drop_duplicates()`

Depois que você **identifica o problema**, você decide: o que fazer?

**3 opções principais:**
1. ❌ **Descartar a linha inteira** (se tiver muitos problemas)
2. ✅ **Preencher o buraco** (se for só um valor faltante — próximo tópico)
3. 🔄 **Substituir/corrigir** (se tiver um erro simples)

#### 🗑️ **Remover Linhas com `dropna()` e `drop_duplicates()`**

In [ ]:
#clientes_final.isna().sum()

# dropna() — remove TODAS as linhas que têm QUALQUER valor faltante
# Cuidado: pode remover MUITAS linhas se tiver muitos NaN



In [ ]:
# dropna() com subset — remove só se ESSA coluna específica tiver NaN

# dropna() com how='all' — remove só se TODAS as colunas forem NaN (raro)


In [ ]:
clientes_final_limpo.isna().sum()

In [ ]:
# Quais são essas linhas duplicadas?


In [ ]:
### JA FOI FALADO ANTERIORMENTE ###

 # drop_duplicates() com subset — considera duplicatas só dessa coluna


# keep='last' — ao invés de manter a primeira cópia, mantém a última


### 4️⃣ Substituição Condicional: `replace()` e `np.where()`

Às vezes você quer **encontrar um padrão errado e corrigir tudo de uma vez**.

**Analogia:** no estoque de um supermercado, você descobriu que todos os "banana" foram cadastrados como "banana prata" quando deveriam ser só "banana". Você quer uma operação que mude todos de uma vez.

#### 🔄 **Substituição Simples com `replace()`**

In [ ]:
clientes_final['estado'].unique()

In [ ]:
# replace() — substitui um valor por outro


In [ ]:
# Substituir VÁRIOS valores de uma vez


In [ ]:
# replace() com regex (procura por padrão)


##### 🎯 **Substituição Condicional com `np.where()`**

Quando a lógica é mais complexa, `np.where()` é melhor.

**Analogia:** você quer classificar clientes por gasto: se gastar > 500, é "VIP", senão é "Normal".

In [ ]:
clientes_final.info()

In [ ]:
# np.where(condição, valor_se_verdade, valor_se_falso)


In [ ]:
# Você pode ANINHAR np.where() para mais categorias


In [ ]:
# np.where() também funciona para corrigir erros


### 3️⃣ Tratamento de Valores Faltantes: `fillna()`

Às vezes remover não é a melhor opção. Você quer **preencher o buraco** com algo sensato.

**Analogia:** você tem um clínica com um formulário. Se um paciente deixa o campo "rua" vazio, você pode:
1. ❌ Jogar fora o paciente (extremo)
2. ✅ Pedir pra ele preencher depois (melhor)
3. ✅ Usar "Não informado" como padrão (razoável)
4. ✅ Usar a rua mais comum da sua base de dados (inteligente)

In [ ]:
clientes_final.isna().sum()

In [ ]:
# fillna() com valor FIXO




In [ ]:
# fillna() com MÉDIA (para números)


# fillna() com MEDIANA (melhor que média se houver atípicos)


In [ ]:
clientes_final.isna().sum()

In [ ]:
# FFILL e BFILL



In [ ]:
# fillna() com ffill (forward fill) — copia o valor ANTERIOR


# fillna() com bfill (backward fill) — copia o valor POSTERIOR


**Quando usar cada um?**
- **Valor fixo:** quando você quer marcar que é "faltante" (ex: "Não informado")
- **Média/mediana:** quando faz sentido numericamente (ex: salário, idade)
- **ffill/bfill:** quando os dados são **séries temporais** (ex: preço diário de uma ação)

---
### 5️⃣ Acesso e Modificação Precisa: `loc[]`, `iloc[]`, `at[]`, `iat[]`

Às vezes você quer **acessar ou modificar uma célula específica** (linha X, coluna Y).

**Analogia:** como encontrar a cadeira quebrada da 3ª fila, 5ª coluna do cinema.

In [ ]:
# loc[] — acessa por RÓTULO (nome da coluna, índice da linha)
# iloc[] — acessa por POSIÇÃO numérica (linha 0, 1, 2...)

# Qual é o valor na linha de índice 100, coluna 'email'?


# ou por posição: 100ª linha, 2ª coluna


In [ ]:
# Modificar uma célula


In [ ]:
# Acessar MÚLTIPLAS linhas e colunas
# Todas as linhas 10 a 50, só as colunas 'nome' e 'email'


In [ ]:
# Primeiras 10 linhas, primeiras 3 colunas


In [ ]:
# Filtrar com CONDIÇÃO usando loc[]
# Todas as linhas onde valor_gasto > 400


In [ ]:
# at[] e iat[] — acesso mais rápido a UMA célula (atalho)


#### ✏️ **Exercícios de Fixação**

#### Exercício 1: Diagnóstico Rápido ⭐ 

**Enunciado:**
Você recebeu um arquivo CSV `alunos.csv` com dados de alunos. Sem abrir em Excel, use Python para:
1. Quantas linhas e colunas o arquivo tem?
2. Quais são os tipos de dados de cada coluna?
3. Qual coluna tem mais valores faltantes?
4. Quantas linhas duplicadas existem?

**Dica:** Use `shape`, `info()`, `isna().sum()`, `duplicated().sum()`




In [52]:
alunos = pd.read_csv('C:/Temp/alunos.csv')
display(alunos)

# 1. Quantas linhas e colunas o arquivo tem?
print(f'O arquivo alunos.csv tem {alunos.shape[0]} linhas e {alunos.shape[1]} colunas.')

# 2. Quais são os tipos de dados de cada coluna?
print("\nTipos de Dados:")
print(alunos.info())

# 3. Qual coluna tem mais valores faltantes?
alunos_nulos = alunos.isna().sum()
print(f'\nA coluna com mais valores nulos é {alunos_nulos.idxmax()} com {max(alunos_nulos)} nulos.')

# 4. Quantas linhas duplicadas existem?
print("\nLinhas Duplicadas", alunos.duplicated().sum())

,id,nome,idade,nota,email
0,1,Ana,20.0,8.5,ana@email.com
1,2,Bruno,22.0,7.0,bruno@email.com
2,3,Carlos,NaN,6.5,NaN
3,4,Daniela,21.0,9.0,daniela@email.com
4,5,Eduardo,23.0,NaN,eduardo@email.com
5,6,Fernanda,20.0,8.0,NaN
6,7,Gabriel,22.0,7.5,gabriel@email.com
7,8,Helena,21.0,8.2,helena@email.com
8,9,Igor,NaN,6.8,NaN
9,10,Julia,20.0,9.1,julia@email.com


O arquivo alunos.csv tem 11 linhas e 5 colunas.

Tipos de Dados:
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      11 non-null     int64  
 1   nome    11 non-null     str    
 2   idade   9 non-null      float64
 3   nota    10 non-null     float64
 4   email   8 non-null      str    
dtypes: float64(2), int64(1), str(2)
memory usage: 762.0 bytes
None

A coluna com mais valores nulos é email com 3 nulos.

Linhas Duplicadas 1


#### Exercício 2: Remover Valores Faltantes Inteligentemente ⭐

**Enunciado:**
O DataFrame `vendas` tem 10 linhas. Coluna 'email' tem 2 valores faltantes (5%), coluna 'telefone' tem 4 faltantes (40%), coluna 'valor' tem 2 faltantes (5%).

Para qual coluna você usa `dropna()` e para qual você usa `fillna()`? Por quê?

```python
# Seu código aqui:
# df = pd.read_csv('vendas.csv')
# ... completar
```

**Dica:** Pense em qual percentual é "aceitável" remover. 5% é OK, 40% é demais.


### Exercício 3: Limpeza Completa com Decisões ⭐⭐ 

**Enunciado:**
Você recebeu `produtos.csv`:

```
id,nome,preco,estoque,categoria,data_cadastro
1,Notebook,R$ 2500,150,Eletrônicos,2024-01-15
1,Notebook,R$ 2500,150,Eletrônicos,2024-01-15
2,Mouse,500,?,Periféricos,2024-01-16
3,Teclado,150.50,80,,2024-01-17
4,Monitor,,50,Eletrônicos,
```

Limpe esse dataset:
1. Remova duplicatas
2. Converta 'preco' de texto para número
3. Trate valores faltantes em 'estoque' (preencha com 0)
4. Trate valores faltantes em 'categoria' (preencha com 'Sem categoria')
5. Mostre as linhas onde 'data_cadastro' é vazia
6. Mostre quantas linhas sobraram

**Dica:** Use `drop_duplicates()`, `str.replace()`, `pd.to_numeric()`, `fillna()`


In [ ]:
# 1. Remove duplicatas do DataFrame correto


In [ ]:
# 2. Converte preco de texto para número


In [ ]:
# 3. Trata estoque — preenche '?' e NaN com 0
#produtos['estoque'] = produtos['estoque'].replace('?', np.nan)  # transforma '?' em NaN


In [ ]:
# 4. Trata categoria — preenche vazio com 'Sem categoria'


In [ ]:
#caso precise fazer para mais de uma coluna



In [ ]:
# 5. Mostre as linhas onde data_cadastro é vazia


In [ ]:
# 6. Linhas resultantes do dataframe


### Exercício 4: Transformação com `apply()` ⭐⭐ 

**Enunciado:**
Você tem `vendas.csv` com coluna 'valor_compra' (em números) e 'estado' (ex: SP, RJ, MG).

Crie uma nova coluna 'imposto':
- Se estado == 'SP': 18% de imposto
- Se estado == 'RJ': 20% de imposto
- Se estado == 'MG': 15% de imposto
- Outros: 17% de imposto

**Dica:** Use `np.where()`

### Exercício 5: Identificar e Corrigir Atípicos ⭐⭐ 

**Enunciado:**
Coluna 'idade' tem valores: `[25, 35, 1500, 28, 0, 42, -5, 50]` (há erros)

Corrija usando `np.where()`:
- Se idade < 18 ou > 100: marque como 'Inválido'
- Senão: mantenha o valor

Depois, remova as inválidas.

In [ ]:
df = pd.DataFrame({'idade': [25, 35, 1500, 28, 0, 42, -5, 50]})

